### Create database 'performance' for the performance indicator and populate it with tables and constants

The demo showcases the story
https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-735


## 1 - Initialisation

In [ ]:
# Access to Prefect
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  

init_demo()

# Reload the global vars again
from resources.utils import *  

In [ ]:
# Other imports
from importlib import reload
import os
import sys
import prefect
from rs_common import prefect_utils
from rs_common.prefect_utils import *
import rs_workflows

# Local paths
rs_workflows_parent = Path(rs_workflows.__path__[0]).parent

# Get the prefect share bucket folder
share_bucket = await get_share_bucket()

## 3 - Deploy Prefect flows

We deploy the Prefect workflows that are implemented in the rs-client-libraries git repository.

WARNING: the rs-client-libraries source code must be identical in these 3 environments:

- https://github.com/RS-PYTHON/rs-demo.git
- This Jupyter environment
- The Prefect Docker images

In [ ]:
%%bash -s "$rs_workflows_parent"
# Deploy the flow
deploy_file=$(realpath "./init_pi_db_flows.yaml")
echo "Deploying '$deploy_file'..."
(cd $1; prefect --no-prompt deploy --prefect-file "$deploy_file" --all)

In [ ]:
# Flow deployment names
pi_deploy = "PI db init/PI db init"
await prefect_utils.wait_for_deployment(pi_deploy)

In [ ]:
# For testing only: serve from a s3 bucket to test changes more easily
debug_flow = False
if debug_flow:

    # Use a subfolder named after the current user
    s3_code_folder = f"users/{OWNER_ID}/code"
    workflows_folder = f"{s3_code_folder}/rs_workflows"
    
    # Upload workflows package and resources contents
    await share_bucket.put_directory(local_path = rs_workflows.__path__[0], to_path = workflows_folder)

    # Reload all rs-client-libraries modules
    for module in list(sys.modules.values()):
        if any(module.__name__.startswith(prefix) for prefix in ["rs_client.", "rs_common.", "rs_workflows."]):
            reload(module)

    # Deploy the flows
    for entrypoint, name, deploy_name in [
        ["init_pi_db_flow.py:on_demand_auxip_staging", "PI db init", pi_deploy],        
    ]:
        flow = await prefect.flow.from_source(
            source=share_bucket,
            entrypoint=f"{workflows_folder}/{entrypoint}",
        )
        await flow.deploy(
            name=name,
            work_pool_name=os.environ["PREFECT_WORK_POOL_EOPF"], 
            tags=["debug only"],
            ignore_warnings=True,
        )
        await prefect_utils.wait_for_deployment(deploy_name)

## 4 - Run flows

Run one flow for CADIP staging and one for AUXIP staging.

In [ ]:
%%bash -s "$pi_deploy"
# Deploy and run flow for CADIP staging
# Trigger a run for this flow from the command line
prefect deployment run "$1" --watch